# 04 · Export to ONNX, quantise to INT8, verify parity

Three things have to hold before anything ships:

* **A9 — parity.** INT8 logits must be within 0.01 of PyTorch's. Quantisation
  that silently costs accuracy is how a model that passed evaluation ships as
  something worse.
* **A11 — bundle size.** Every function under 500 MB.
* **Graph contract.** Inputs named exactly `input_ids`, `attention_mask`,
  `features`. `api/_lib/classifier.py` feeds those names and refuses to guess,
  so a renamed input degrades to "model unavailable" rather than wrong
  scores — but catching it here is far better.

Runtime: 15–30 minutes. CPU only.


In [ ]:
# Kaggle setup. Run once per session.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow

import sys, os
from pathlib import Path

# The repo is added as a Kaggle dataset, or cloned. Point REPO at it.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
print("repo:", REPO)


In [ ]:
import numpy as np, pandas as pd, torch, json
from pathlib import Path
from transformers import AutoTokenizer

from lib.data import FEATURE_NAMES
from lib.export import (check_parity, export_onnx, export_tokenizer,
                        quantize, report_sizes, write_manifest)
from lib.model import Detector, DetectorConfig

def load(tag):
    state = torch.load(WORK / f"model_{tag}" / "final.pt", map_location="cpu")
    config = state["config"]
    model = Detector(DetectorConfig(backbone=config["backbone"], max_length=config["max_length"]))
    model.load_state_dict(state["model"])
    model.eval()
    return model, config


In [ ]:
parity = {}

for tag in ("a", "b"):
    print(f"=== Model {tag.upper()}")
    model, config = load(tag)
    tokenizer = AutoTokenizer.from_pretrained(config["backbone"], use_fast=True)

    fp32 = export_onnx(model, MODELS / f"detector_{tag}_fp32.onnx",
                       max_length=config["max_length"])
    print("  exported, graph contract verified")

    int8 = quantize(fp32, MODELS / f"detector_{tag}_int8.onnx")
    print(f"  fp32 {fp32.stat().st_size >> 20} MB -> int8 {int8.stat().st_size >> 20} MB")

    export_tokenizer(tokenizer, MODELS / f"detector_{tag}_tokenizer.json")

    # A9: parity on real held-out text, standardised with this model's own
    # training statistics.
    sample = pd.read_parquet(DATA / f"train_{tag}.parquet").sample(32, random_state=0)
    mean = np.asarray(config["feature_mean"]); std = np.asarray(config["feature_std"])
    features = ((sample[list(FEATURE_NAMES)].to_numpy(np.float64) - mean) / std).astype(np.float32)

    parity[tag] = check_parity(model, int8, tokenizer, sample["text"].tolist(),
                               features, config["max_length"])
    print(f"  max logit delta {parity[tag]['max_logit_delta']:.5f} "
          f"({'PASS' if parity[tag]['passes_a9'] else 'FAIL'} A9)")

    fp32.unlink()   # only the quantised graph ships


In [ ]:
# The base models. No training needed, so a fresh clone has a working
# Binoculars signal and similarity gate immediately.
!cd $REPO && python scripts/fetch_models.py --base-models
!cp -n $REPO/models/*.onnx $MODELS/ 2>/dev/null; cp -n $REPO/models/*tokenizer.json $MODELS/ 2>/dev/null


In [ ]:
# A11: every function bundle under 500 MB.
sizes = report_sizes(MODELS)
for name, mb in sizes.items():
    print(f"  {name:22s} {mb:6.1f} MB   {'OVER LIMIT' if mb > 500 else 'ok'}")

assert all(mb <= 500 for mb in sizes.values()), "A11 fails — a bundle is over 500 MB"

write_manifest(MODELS, {
    "parity": parity,
    "bundle_sizes_mb": sizes,
    "opset": 17,
    "quantization": "dynamic int8",
})
print("\nA9:", all(p["passes_a9"] for p in parity.values()))


## Publish

Upload `MODELS` to a **public** Hugging Face repo — free and unmetered, unlike
GitHub LFS (1 GB storage / 1 GB bandwidth per month, which repeated deploys
would exhaust in days).

```python
from huggingface_hub import HfApi
api = HfApi(token="hf_...")
api.create_repo("YOUR_NAME/ai-text-detector-onnx", exist_ok=True)
api.upload_folder(folder_path=str(MODELS), repo_id="YOUR_NAME/ai-text-detector-onnx")
```

Then set `MODEL_REPO=YOUR_NAME/ai-text-detector-onnx` in the Vercel project
and redeploy. `scripts/fetch_models.py` picks it up at build time.
